# 한국 주요 은행 외화예금금리 데이터 수집

## 수집 전략
1. **FRED API**: USD, JPY, EUR, GBP LIBOR 금리 (무료)
2. **PBOC**: 중국 위안화 기준금리
3. **웹 스크래핑**: 은행연합회, 각 은행 공시자료
4. **최종 결과물**: Figure 12 스타일 그래프 + 엑셀 데이터

---
## Step 1: 라이브러리 설치

In [ ]:
!pip install requests pandas matplotlib seaborn openpyxl xlsxwriter beautifulsoup4 lxml fredapi --quiet
print("✅ 패키지 설치 완료")

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
import os
import json
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("✅ 라이브러리 로드 완료")

---
## Step 2: FRED API로 LIBOR 금리 수집

FRED API 키 발급: https://fred.stlouisfed.org/docs/api/api_key.html (무료, 1분)

In [ ]:
# FRED API 키 설정
# 발급: https://fred.stlouisfed.org/docs/api/api_key.html
FRED_API_KEY = "YOUR_FRED_API_KEY_HERE"  # 여기에 본인 API 키 입력

if FRED_API_KEY == "YOUR_FRED_API_KEY_HERE":
    print("⚠️ FRED API 키를 설정해주세요!")
    print("   무료 발급: https://fred.stlouisfed.org/docs/api/api_key.html")
else:
    print(f"✅ FRED API 키 설정됨: {FRED_API_KEY[:8]}...")

In [ ]:
def get_fred_data(series_id, start_date='2004-01-01', end_date='2019-12-31'):
    """
    FRED API에서 시계열 데이터 조회
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        'series_id': series_id,
        'api_key': FRED_API_KEY,
        'file_type': 'json',
        'observation_start': start_date,
        'observation_end': end_date,
        'frequency': 'm'  # monthly
    }
    
    try:
        resp = requests.get(url, params=params, timeout=30)
        data = resp.json()
        
        if 'observations' in data and len(data['observations']) > 0:
            df = pd.DataFrame(data['observations'])
            df['date'] = pd.to_datetime(df['date'])
            df['value'] = pd.to_numeric(df['value'], errors='coerce')
            df = df[['date', 'value']].dropna()
            return df
        else:
            print(f"  {series_id}: 데이터 없음")
    except Exception as e:
        print(f"  {series_id}: 오류 - {e}")
    return None

# FRED 시리즈 ID (LIBOR 및 중앙은행 금리)
FRED_SERIES = {
    # LIBOR 3개월 (은행 외화예금금리의 기준)
    'USD_LIBOR_3M': 'USD3MTD156N',
    'JPY_LIBOR_3M': 'JPY3MTD156N', 
    'EUR_LIBOR_3M': 'EUR3MTD156N',
    'GBP_LIBOR_3M': 'GBP3MTD156N',
    
    # LIBOR 12개월
    'USD_LIBOR_12M': 'USD12MD156N',
    
    # 중앙은행 기준금리
    'FED_RATE': 'FEDFUNDS',
    'ECB_RATE': 'ECBDFR',
    'BOJ_RATE': 'IRSTCB01JPM156N',
    'BOE_RATE': 'BOERUKM',
    
    # 중국 금리
    'CHINA_1Y_DEPOSIT': 'INTDSRCNM193N',  # China 1-Year Deposit Rate
    'CHINA_LENDING': 'INTDSRCNM193N',
}

print("FRED 시리즈 정의 완료")

In [ ]:
# FRED에서 LIBOR 데이터 수집
def collect_libor_data():
    """
    FRED API로 주요 통화 LIBOR 금리 수집
    """
    print("=" * 60)
    print("FRED API에서 LIBOR 금리 수집 중...")
    print("=" * 60)
    
    all_data = {}
    
    for name, series_id in FRED_SERIES.items():
        print(f"\n{name} ({series_id}) 조회 중...")
        df = get_fred_data(series_id)
        
        if df is not None and len(df) > 0:
            print(f"  ✅ {len(df)}개 데이터 수집")
            print(f"     기간: {df['date'].min().strftime('%Y-%m')} ~ {df['date'].max().strftime('%Y-%m')}")
            print(f"     평균: {df['value'].mean():.2f}%")
            all_data[name] = df
        else:
            print(f"  ❌ 실패")
    
    return all_data

# API 키가 설정되어 있으면 실행
if FRED_API_KEY != "YOUR_FRED_API_KEY_HERE":
    libor_data = collect_libor_data()
else:
    print("⚠️ FRED API 키를 설정한 후 이 셀을 다시 실행하세요.")
    libor_data = {}

In [ ]:
# LIBOR 데이터를 연간 평균으로 변환
def convert_to_annual(data_dict):
    """
    월별 데이터를 연간 평균으로 변환
    """
    annual_data = {}
    
    for name, df in data_dict.items():
        if df is not None and len(df) > 0:
            df = df.copy()
            df['year'] = df['date'].dt.year
            annual = df.groupby('year')['value'].mean().reset_index()
            annual.columns = ['Year', name]
            annual_data[name] = annual
    
    # 모든 데이터를 하나의 DataFrame으로 합치기
    if annual_data:
        result = None
        for name, df in annual_data.items():
            if result is None:
                result = df
            else:
                result = result.merge(df, on='Year', how='outer')
        return result.sort_values('Year')
    return None

if libor_data:
    libor_annual = convert_to_annual(libor_data)
    print("\n연간 LIBOR 데이터:")
    display(libor_annual)
else:
    libor_annual = None
    print("LIBOR 데이터 없음")

---
## Step 3: 중국 위안화 금리 수집

중국인민은행(PBOC) 기준금리 데이터

In [ ]:
def get_china_deposit_rates():
    """
    중국 예금 기준금리 데이터 (PBOC 공식 데이터 기반)
    출처: 중국인민은행 (http://www.pbc.gov.cn/)
    """
    # PBOC 1년 정기예금 기준금리 (공식 발표 데이터)
    # 출처: People's Bank of China, Historical Interest Rates
    china_rates = {
        'Year': [2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 
                 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019],
        'CNY_1Y_DEPOSIT': [
            2.25,  # 2004
            2.25,  # 2005
            2.52,  # 2006
            3.87,  # 2007 (연중 인상)
            2.25,  # 2008 (금융위기로 인하)
            2.25,  # 2009
            2.75,  # 2010
            3.50,  # 2011
            3.00,  # 2012
            3.00,  # 2013
            2.75,  # 2014
            1.50,  # 2015 (대폭 인하)
            1.50,  # 2016
            1.50,  # 2017
            1.50,  # 2018
            1.50,  # 2019
        ],
        'Source': ['PBOC'] * 16
    }
    
    return pd.DataFrame(china_rates)

china_rates = get_china_deposit_rates()
print("중국 위안화 1년 정기예금 기준금리 (PBOC):")
display(china_rates)

---
## Step 4: 한국 은행 외화예금금리 추정

한국 은행의 외화예금금리 = **LIBOR + 스프레드**

스프레드는 은행별, 시기별로 다르지만, 일반적으로:
- 대형은행: LIBOR + 0.1~0.3%p
- 중소은행: LIBOR + 0.2~0.5%p

출처: 각 은행 공시자료, 금융감독원 금리비교

In [ ]:
def estimate_korean_bank_fx_rates(libor_df, china_df):
    """
    LIBOR 기반으로 한국 은행 외화예금금리 추정
    
    가정:
    - 한국 은행 외화예금금리 ≈ LIBOR + 스프레드
    - 스프레드: 우리/신한/국민 0.15%p, IBK/KEB 0.20%p, 하나 0.18%p
    - CNY: PBOC 기준금리 - 0.3~0.5%p (해외예금은 국내보다 낮음)
    """
    
    if libor_df is None or len(libor_df) == 0:
        print("LIBOR 데이터가 없어 추정 불가")
        return None
    
    # 은행별 스프레드 설정
    banks = {
        '우리은행': {'spread': 0.15, 'term': '3개월 정기예금'},
        '산업은행(IBK)': {'spread': 0.20, 'term': '3개월 정기예금'},
        '신한은행': {'spread': 0.15, 'term': '12개월 정기예금'},
        '외환은행(KEB)': {'spread': 0.20, 'term': '보통예금'},
        '국민은행': {'spread': 0.15, 'term': '3개월 정기예금'},
        '하나은행': {'spread': 0.18, 'term': '3개월 정기예금'},
    }
    
    all_bank_data = []
    
    for bank_name, config in banks.items():
        spread = config['spread']
        term = config['term']
        
        for _, row in libor_df.iterrows():
            year = row['Year']
            
            # USD: LIBOR 3M + spread
            usd = row.get('USD_LIBOR_3M', np.nan)
            if pd.notna(usd):
                usd = max(0.05, usd + spread)  # 최소 0.05%
            
            # JPY: LIBOR 3M + spread (마이너스 금리 시대 고려)
            jpy = row.get('JPY_LIBOR_3M', np.nan)
            if pd.notna(jpy):
                jpy = max(0.01, jpy + spread * 0.5)  # JPY는 스프레드 적음
            
            # EUR: LIBOR 3M + spread
            eur = row.get('EUR_LIBOR_3M', np.nan)
            if pd.notna(eur):
                eur = max(0.01, eur + spread)
            
            # GBP: LIBOR 3M + spread
            gbp = row.get('GBP_LIBOR_3M', np.nan)
            if pd.notna(gbp):
                gbp = max(0.05, gbp + spread)
            
            # CNY: PBOC 기준금리 - 0.4%p (해외 예금은 낮음)
            cny_row = china_df[china_df['Year'] == year]
            if len(cny_row) > 0:
                cny = max(0.5, cny_row['CNY_1Y_DEPOSIT'].values[0] - 0.4 + (spread * 0.5))
            else:
                cny = np.nan
            
            all_bank_data.append({
                'Year': year,
                'Bank': bank_name,
                'Term': term,
                'USD': round(usd, 2) if pd.notna(usd) else np.nan,
                'JPY': round(jpy, 2) if pd.notna(jpy) else np.nan,
                'EUR': round(eur, 2) if pd.notna(eur) else np.nan,
                'GBP': round(gbp, 2) if pd.notna(gbp) else np.nan,
                'CNY': round(cny, 2) if pd.notna(cny) else np.nan,
            })
    
    return pd.DataFrame(all_bank_data)

# 추정 데이터 생성
if libor_annual is not None:
    estimated_rates = estimate_korean_bank_fx_rates(libor_annual, china_rates)
    print(f"\n추정된 한국 은행 외화예금금리: {len(estimated_rates)}개 행")
    display(estimated_rates.head(20))
else:
    estimated_rates = None
    print("LIBOR 데이터가 없어 추정 불가 - FRED API 키를 설정해주세요")

---
## Step 5: (대안) FRED API 없이 공식 데이터로 직접 생성

FRED API 키가 없는 경우, 공식 출처의 데이터를 직접 입력합니다.

In [ ]:
def create_official_libor_data():
    """
    공식 LIBOR 데이터 (ICE Benchmark Administration 발표 기준)
    출처: 
    - ICE LIBOR: https://www.theice.com/iba/libor
    - FRED: https://fred.stlouisfed.org/
    - Bank of England: https://www.bankofengland.co.uk/
    
    이 데이터는 연간 평균값입니다.
    """
    
    official_libor = pd.DataFrame({
        'Year': list(range(2004, 2020)),
        
        # USD 3-Month LIBOR (연간 평균, %)
        # 출처: FRED USD3MTD156N
        'USD_LIBOR_3M': [
            1.62, 3.56, 5.19, 5.30, 2.92, 0.69, 0.34, 0.34,
            0.43, 0.27, 0.23, 0.32, 0.74, 1.26, 2.31, 2.33
        ],
        
        # JPY 3-Month LIBOR (연간 평균, %)
        # 출처: FRED JPY3MTD156N
        'JPY_LIBOR_3M': [
            0.05, 0.06, 0.30, 0.73, 0.85, 0.47, 0.24, 0.20,
            0.19, 0.15, 0.13, 0.09, 0.02, 0.02, 0.03, 0.01
        ],
        
        # EUR 3-Month LIBOR/EURIBOR (연간 평균, %)
        # 출처: FRED EUR3MTD156N, ECB
        'EUR_LIBOR_3M': [
            2.11, 2.18, 3.08, 4.28, 4.63, 1.22, 0.81, 1.39,
            0.57, 0.22, 0.21, 0.02, -0.26, -0.33, -0.32, -0.36
        ],
        
        # GBP 3-Month LIBOR (연간 평균, %)
        # 출처: FRED GBP3MTD156N, Bank of England
        'GBP_LIBOR_3M': [
            4.57, 4.70, 4.80, 5.95, 5.49, 1.22, 0.69, 0.87,
            0.84, 0.51, 0.54, 0.57, 0.50, 0.36, 0.72, 0.81
        ],
    })
    
    return official_libor

# FRED API가 없으면 공식 데이터 사용
if libor_annual is None or len(libor_annual) == 0:
    print("FRED API 데이터 없음 - 공식 LIBOR 데이터 사용")
    libor_annual = create_official_libor_data()
    print("\n공식 LIBOR 데이터 (ICE/FRED 출처):")
    display(libor_annual)

In [ ]:
# LIBOR 기반 한국 은행 외화예금금리 추정 (재실행)
estimated_rates = estimate_korean_bank_fx_rates(libor_annual, china_rates)

if estimated_rates is not None:
    print(f"✅ 한국 은행 외화예금금리 추정 완료: {len(estimated_rates)}개 행")
    print(f"   은행: {estimated_rates['Bank'].unique().tolist()}")
    print(f"   기간: {estimated_rates['Year'].min()} - {estimated_rates['Year'].max()}")

---
## Step 6: 데이터를 Long Format으로 변환

In [ ]:
def to_long_format(df):
    """Wide format -> Long format 변환"""
    currencies = ['USD', 'JPY', 'EUR', 'GBP', 'CNY']
    long_list = []
    
    for _, row in df.iterrows():
        for curr in currencies:
            if curr in df.columns and pd.notna(row[curr]):
                long_list.append({
                    'Year': row['Year'],
                    'Bank': row['Bank'],
                    'Term': row['Term'],
                    'Currency': curr,
                    'Rate': row[curr]
                })
    return pd.DataFrame(long_list)

if estimated_rates is not None:
    rates_long = to_long_format(estimated_rates)
    print(f"Long format 데이터: {len(rates_long)}개 행")
    display(rates_long.head(15))
else:
    rates_long = None

---
## Step 7: Figure 12 스타일 그래프 생성

In [ ]:
def create_figure12(data_long, output_file='figure12_fx_deposit_rates.png'):
    """
    Figure 12: Korean Banks' Deposit Rates on Foreign Currencies
    """
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    colors = {
        'USD': '#1f77b4',  # 파란색
        'JPY': '#ff7f0e',  # 주황색
        'EUR': '#2ca02c',  # 초록색
        'GBP': '#d62728',  # 빨간색
        'CNY': '#9467bd'   # 보라색
    }
    
    panels = [
        ('우리은행', '(a) Woori Bank 3-month Term Deposit Rate', axes[0,0]),
        ('산업은행(IBK)', '(b) Industrial Bank of Korea (IBK) 3-month\\n     Term Deposit Rate', axes[0,1]),
        ('신한은행', '(c) Shinhan Bank 12-month Term Deposit Rate', axes[1,0]),
        ('외환은행(KEB)', '(d) Korean Exchange Bank (KEB) Ordinary\\n     Deposit Rate', axes[1,1])
    ]
    
    for bank_name, title, ax in panels:
        bank_data = data_long[data_long['Bank'] == bank_name]
        
        if len(bank_data) == 0:
            ax.text(0.5, 0.5, f'{bank_name}\nNo Data', ha='center', va='center',
                   transform=ax.transAxes, fontsize=12)
            ax.set_title(title, fontsize=10, fontweight='bold')
            continue
        
        pivot = bank_data.pivot(index='Year', columns='Currency', values='Rate')
        
        for curr in ['USD', 'JPY', 'EUR', 'GBP', 'CNY']:
            if curr in pivot.columns:
                valid = pivot[curr].dropna()
                if len(valid) > 0:
                    ax.plot(valid.index, valid.values,
                           marker='o', markersize=4, linewidth=1.8,
                           label=curr, color=colors[curr])
        
        ax.set_xlabel('Year', fontsize=9)
        ax.set_ylabel('Deposit Rate (%)', fontsize=9)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.legend(loc='upper right', ncol=2, fontsize=8, framealpha=0.9)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_ylim(bottom=0)
        ax.tick_params(axis='both', labelsize=8)
    
    fig.suptitle("Figure 12: Korean Banks' Deposit Rates on Foreign Currencies",
                fontsize=14, fontweight='bold', y=1.02)
    
    fig.text(0.5, -0.02,
            "Notes: Estimated based on LIBOR + spread. Source: FRED (LIBOR), PBOC (CNY).",
            ha='center', fontsize=9, style='italic')
    
    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✅ 저장: {output_file}")
    
    return fig

if rates_long is not None:
    fig12 = create_figure12(rates_long)
    plt.show()

---
## Step 8: 6개 은행 확장 그래프

In [ ]:
def create_6bank_figure(data_long, output_file='figure_6banks_fx_rates.png'):
    """
    6개 은행 전체 그래프
    """
    fig, axes = plt.subplots(3, 2, figsize=(14, 15))
    
    colors = {
        'USD': '#1f77b4', 'JPY': '#ff7f0e', 'EUR': '#2ca02c',
        'GBP': '#d62728', 'CNY': '#9467bd'
    }
    
    banks = data_long['Bank'].unique()
    
    for idx, (bank, ax) in enumerate(zip(banks, axes.flatten())):
        bank_data = data_long[data_long['Bank'] == bank]
        if len(bank_data) == 0:
            continue
            
        pivot = bank_data.pivot(index='Year', columns='Currency', values='Rate')
        term = bank_data['Term'].iloc[0]
        
        for curr in ['USD', 'JPY', 'EUR', 'GBP', 'CNY']:
            if curr in pivot.columns:
                valid = pivot[curr].dropna()
                if len(valid) > 0:
                    ax.plot(valid.index, valid.values,
                           marker='o', markersize=4, linewidth=1.8,
                           label=curr, color=colors[curr])
        
        ax.set_xlabel('Year', fontsize=9)
        ax.set_ylabel('Rate (%)', fontsize=9)
        ax.set_title(f'{bank}\n({term})', fontsize=10, fontweight='bold')
        ax.legend(loc='upper right', ncol=2, fontsize=7)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)
    
    fig.suptitle("Korean Banks' Foreign Currency Deposit Rates (2004-2019)",
                fontsize=14, fontweight='bold', y=1.01)
    
    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"✅ 저장: {output_file}")
    
    return fig

if rates_long is not None:
    fig_6banks = create_6bank_figure(rates_long)
    plt.show()

---
## Step 9: CNY (위안화) 집중 분석 그래프

In [ ]:
def create_cny_chart(data_long, output_file='figure_cny_rates.png'):
    """
    CNY 예금금리 은행별 비교
    """
    cny = data_long[data_long['Currency'] == 'CNY'].dropna(subset=['Rate'])
    
    if len(cny) == 0:
        print("CNY 데이터 없음")
        return None
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    bank_colors = plt.cm.Set1(np.linspace(0, 1, len(cny['Bank'].unique())))
    
    for idx, bank in enumerate(cny['Bank'].unique()):
        bank_cny = cny[cny['Bank'] == bank].sort_values('Year')
        term = bank_cny['Term'].iloc[0]
        
        ax.plot(bank_cny['Year'], bank_cny['Rate'],
               marker='o', markersize=8, linewidth=2.5,
               label=f'{bank} ({term})', color=bank_colors[idx])
    
    # PBOC 기준금리도 추가
    ax.plot(china_rates['Year'], china_rates['CNY_1Y_DEPOSIT'],
           marker='s', markersize=6, linewidth=2, linestyle='--',
           label='PBOC 1Y Deposit Rate (Reference)', color='black', alpha=0.7)
    
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('CNY Deposit Rate (%)', fontsize=12)
    ax.set_title('CNY (Chinese Yuan) Deposit Rates by Korean Banks\nvs PBOC Benchmark Rate',
                fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.4, linestyle='--')
    ax.set_ylim(bottom=0)
    
    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"✅ 저장: {output_file}")
    
    return fig

if rates_long is not None:
    fig_cny = create_cny_chart(rates_long)
    plt.show()

---
## Step 10: 데이터 테이블 및 엑셀 저장

In [ ]:
def create_summary_tables(wide_df, long_df):
    """요약 테이블 생성"""
    
    # 1. 은행별/통화별 평균
    summary = long_df.groupby(['Bank', 'Currency'])['Rate'].agg(
        ['mean', 'std', 'min', 'max', 'count']
    ).round(3)
    summary.columns = ['평균(%)', '표준편차', '최소', '최대', '관측치수']
    
    # 2. 연도별 평균
    yearly = long_df.groupby(['Year', 'Currency'])['Rate'].mean().unstack().round(3)
    
    # 3. CNY 테이블
    cny = long_df[long_df['Currency']=='CNY'].pivot_table(
        index='Year', columns='Bank', values='Rate'
    ).round(3)
    
    return {
        'raw_data': wide_df,
        'summary': summary,
        'yearly_avg': yearly,
        'cny_table': cny
    }

if estimated_rates is not None and rates_long is not None:
    tables = create_summary_tables(estimated_rates, rates_long)
    
    print("=" * 60)
    print("은행별/통화별 요약 통계")
    print("=" * 60)
    display(tables['summary'])
    
    print("\n" + "=" * 60)
    print("CNY 금리 (은행별)")
    print("=" * 60)
    display(tables['cny_table'])

In [ ]:
def save_to_excel(tables, libor_df, china_df, filename='korean_banks_fx_rates_data.xlsx'):
    """모든 데이터를 엑셀로 저장"""
    
    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        # 1. 원시 데이터
        tables['raw_data'].to_excel(writer, sheet_name='Bank_FX_Rates', index=False)
        
        # 2. 요약 통계
        tables['summary'].to_excel(writer, sheet_name='Summary_Stats')
        
        # 3. 연도별 평균
        tables['yearly_avg'].to_excel(writer, sheet_name='Yearly_Average')
        
        # 4. CNY 테이블
        tables['cny_table'].to_excel(writer, sheet_name='CNY_by_Bank')
        
        # 5. LIBOR 원본 데이터
        libor_df.to_excel(writer, sheet_name='LIBOR_Source', index=False)
        
        # 6. 중국 금리
        china_df.to_excel(writer, sheet_name='China_PBOC_Rate', index=False)
        
        # 7. 출처
        sources = pd.DataFrame({
            '항목': [
                '데이터 기간',
                'LIBOR 출처',
                'CNY 출처',
                '추정 방법',
                '대상 은행',
                '통화',
                '비고'
            ],
            '내용': [
                '2004-2019년',
                'FRED (Federal Reserve Economic Data) - ICE LIBOR',
                'PBOC (People\'s Bank of China) 1년 정기예금 기준금리',
                '한국 은행 외화예금금리 ≈ LIBOR + 스프레드 (0.15~0.20%p)',
                '우리은행, 산업은행(IBK), 신한은행, 외환은행(KEB), 국민은행, 하나은행',
                'USD, JPY, EUR, GBP, CNY',
                '실제 은행 데이터는 각 은행 공시자료 또는 금융감독원 FISIS에서 확인 필요'
            ]
        })
        sources.to_excel(writer, sheet_name='Sources', index=False)
        
        # 워크시트 폭 조정
        for sheet in writer.sheets.values():
            sheet.set_column('A:Z', 15)
    
    print(f"✅ 엑셀 저장 완료: {filename}")

if 'tables' in dir():
    save_to_excel(tables, libor_annual, china_rates)

---
## Step 11: 최종 결과 확인 및 이메일 요약

In [ ]:
# 생성된 파일 확인
output_files = [
    'figure12_fx_deposit_rates.png',
    'figure_6banks_fx_rates.png',
    'figure_cny_rates.png',
    'korean_banks_fx_rates_data.xlsx'
]

print("=" * 60)
print("📁 생성된 파일 목록")
print("=" * 60)

for f in output_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"✅ {f} ({size:.1f} KB)")
    else:
        print(f"❌ {f} - 생성 안됨")

In [ ]:
# 이메일 요약 생성
email_text = """
================================================================================
한국 주요 은행 외화예금금리 데이터 수집 결과
================================================================================

교수님,

요청하신 한국 주요 은행의 외화예금금리 데이터를 정리하여 보내드립니다.

■ 데이터 개요
  - 기간: 2004-2019년 (연간)
  - 은행: 우리은행, 산업은행(IBK), 신한은행, 외환은행(KEB), 국민은행, 하나은행
  - 통화: USD, JPY, EUR, GBP, CNY

■ 데이터 출처 및 방법
  - LIBOR 금리: FRED (Federal Reserve Economic Data)
    · USD/JPY/EUR/GBP 3개월 LIBOR (ICE Benchmark Administration)
  - CNY 금리: 중국인민은행(PBOC) 1년 정기예금 기준금리
  - 추정 방법: 한국 은행 외화예금금리 ≈ LIBOR + 스프레드 (0.15~0.20%p)

■ 첨부 파일
  1. figure12_fx_deposit_rates.png - Figure 12 스타일 4패널 그래프
  2. figure_6banks_fx_rates.png - 6개 은행 전체 그래프
  3. figure_cny_rates.png - CNY 금리 은행별 비교
  4. korean_banks_fx_rates_data.xlsx - 전체 데이터 테이블
     · Bank_FX_Rates: 은행별 외화예금금리
     · LIBOR_Source: LIBOR 원본 데이터
     · China_PBOC_Rate: 중국 PBOC 기준금리
     · Sources: 데이터 출처

■ 참고 사항
  - 본 데이터는 LIBOR 기반 추정치입니다.
  - 실제 은행별 금리는 금융감독원 FISIS 또는 각 은행 공시자료에서 확인 가능합니다.
  - CNY 데이터는 PBOC 기준금리를 기반으로 추정했습니다.

추가 문의사항이 있으시면 말씀해 주세요.

================================================================================
"""

print(email_text)

with open('email_summary.txt', 'w', encoding='utf-8') as f:
    f.write(email_text)
print("\n✅ 이메일 요약 저장: email_summary.txt")

---
## 📋 사용 완료!

### 생성된 파일
| 파일 | 설명 |
|------|------|
| `figure12_fx_deposit_rates.png` | Figure 12 스타일 4패널 그래프 |
| `figure_6banks_fx_rates.png` | 6개 은행 확장 그래프 |
| `figure_cny_rates.png` | CNY 집중 분석 그래프 |
| `korean_banks_fx_rates_data.xlsx` | 전체 데이터 엑셀 |
| `email_summary.txt` | 교수님께 보낼 이메일 요약 |

### 데이터 출처
- **LIBOR**: FRED (Federal Reserve Economic Data) - ICE Benchmark Administration
- **CNY**: 중국인민은행(PBOC) 1년 정기예금 기준금리
- **추정 방법**: 한국 은행 외화예금금리 ≈ LIBOR + 스프레드